# Direction-Only Classification (Zero-Shot)

Can the emotion direction vectors themselves serve as a classifier — without the trained linear head? This notebook projects each test embedding onto every emotion direction and assigns the class with the largest projection. If this simple geometric classifier is competitive with the trained head, it demonstrates that the model's emotion knowledge **is** the directions, not learned classifier weights.

**Key question:** How much of the trained classifier's performance is recoverable from direction geometry alone?

In [ ]:
from __future__ import annotations

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)


def run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)


def ensure_packages() -> None:
    package_map = {
        "yaml": "pyyaml", "pandas": "pandas", "numpy": "numpy",
        "matplotlib": "matplotlib", "seaborn": "seaborn",
        "torch": "torch", "transformers": "transformers",
        "sklearn": "scikit-learn", "tqdm": "tqdm",
    }
    missing = sorted({pkg for module, pkg in package_map.items() if importlib.util.find_spec(module) is None})
    if missing:
        run_command([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Required packages are already available.")


def find_local_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, cwd / "FinalProject"]:
        candidate = candidate.resolve()
        if (candidate / "configs" / "wav2vec.yaml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root locally.")


def repo_status_lines(repo_root: Path) -> list[str]:
    result = subprocess.run(["git", "-C", str(repo_root), "status", "--short"], text=True, capture_output=True, check=True)
    return [line for line in result.stdout.splitlines() if line.strip()]


def repo_ahead_behind(repo_root: Path) -> tuple[int, int]:
    result = subprocess.run(["git", "-C", str(repo_root), "rev-list", "--left-right", "--count", "HEAD...origin/main"], text=True, capture_output=True, check=False)
    if result.returncode != 0 or not result.stdout.strip():
        return 0, 0
    ahead_str, behind_str = result.stdout.strip().split()
    return int(ahead_str), int(behind_str)


def clone_clean_code_checkout(clean_root: Path) -> Path:
    if clean_root.exists():
        shutil.rmtree(clean_root)
    run_command(["git", "clone", "--depth", "1", REPO_URL, str(clean_root)])
    return clean_root


def prepare_project_and_code_roots() -> tuple[Path, Path]:
    runtime_root = Path("/content") / REPO_NAME
    clean_root = Path("/content") / f"{REPO_NAME}-clean"
    if runtime_root.exists() and (runtime_root / ".git").exists():
        try:
            run_command(["git", "-C", str(runtime_root), "fetch", "origin"])
        except subprocess.CalledProcessError:
            pass
        status_lines = repo_status_lines(runtime_root)
        ahead, behind = repo_ahead_behind(runtime_root)
        if not status_lines and ahead == 0:
            if behind > 0:
                try:
                    run_command(["git", "-C", str(runtime_root), "pull", "--ff-only", "origin", "main"])
                    code_root = runtime_root
                except subprocess.CalledProcessError:
                    code_root = clone_clean_code_checkout(clean_root)
            else:
                code_root = runtime_root
        else:
            code_root = clone_clean_code_checkout(clean_root)
        project_root = runtime_root
    elif runtime_root.exists():
        project_root = runtime_root
        code_root = clone_clean_code_checkout(clean_root)
    else:
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(runtime_root)])
        project_root = runtime_root
        code_root = runtime_root
    return project_root, code_root


ensure_packages()

if IS_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT, CODE_ROOT = prepare_project_and_code_roots()
else:
    PROJECT_ROOT = find_local_project_root()
    CODE_ROOT = PROJECT_ROOT

os.chdir(PROJECT_ROOT)
for root in [str(CODE_ROOT), str(PROJECT_ROOT)]:
    while root in sys.path:
        sys.path.remove(root)
sys.path.insert(0, str(CODE_ROOT))
if str(PROJECT_ROOT) != str(CODE_ROOT):
    sys.path.insert(1, str(PROJECT_ROOT))

stale_modules = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
for name in stale_modules:
    del sys.modules[name]

print("Project root:", PROJECT_ROOT)
print("Code root:", CODE_ROOT)

In [ ]:
from pathlib import Path

experiment_name = 'wav2vec_ravdess_speaker_independent_v1'
local_checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / experiment_name
local_analysis_dir = PROJECT_ROOT / 'artifacts' / 'analysis' / experiment_name
local_output_dir = local_analysis_dir / 'direction_classification'
local_output_dir.mkdir(parents=True, exist_ok=True)

drive_run_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / experiment_name if IS_COLAB else None
drive_checkpoint_dir = drive_run_root / 'checkpoint' if drive_run_root else None
drive_analysis_dir = drive_run_root / 'analysis' if drive_run_root else None
drive_output_dir = drive_analysis_dir / 'direction_classification' if drive_analysis_dir else None

checkpoint_dir = local_checkpoint_dir
if not (checkpoint_dir / 'model_state.pt').exists() and drive_checkpoint_dir is not None and (drive_checkpoint_dir / 'model_state.pt').exists():
    checkpoint_dir = drive_checkpoint_dir

analysis_source_dir = local_analysis_dir
if not (analysis_source_dir / 'embedding_arrays.npz').exists() and drive_analysis_dir is not None and (drive_analysis_dir / 'embedding_arrays.npz').exists():
    analysis_source_dir = drive_analysis_dir

metadata_source_path = analysis_source_dir / 'ravdess_metadata_resolved.csv'
if not metadata_source_path.exists():
    metadata_source_path = PROJECT_ROOT / 'data' / 'metadata' / 'ravdess_metadata.csv'

required_files = [
    checkpoint_dir / 'model_state.pt',
    analysis_source_dir / 'embedding_arrays.npz',
    analysis_source_dir / 'embedding_metadata.csv',
]
missing = [str(p) for p in required_files if not p.exists()]
assert not missing, 'Missing inputs: ' + ', '.join(missing)

print('Checkpoint source:', checkpoint_dir)
print('Analysis source:', analysis_source_dir)
print('Output directory:', local_output_dir)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from src.analysis.emotion_vectors import (
    build_direction_vectors,
    compute_class_centroids,
    load_embedding_artifacts,
)
from src.analysis.anthropic_style import build_train_directions_with_controls
from src.analysis.advanced_analysis import (
    direction_classify,
    evaluate_direction_classifier,
    build_direction_classifier_comparison,
)

artifacts = load_embedding_artifacts(analysis_source_dir)
source_metadata = pd.read_csv(metadata_source_path)
merge_cols = [c for c in ['file_name', 'repetition_code', 'repetition'] if c in source_metadata.columns]
metadata = artifacts.metadata.merge(source_metadata[merge_cols].drop_duplicates(subset=['file_name']), on='file_name', how='left')

label_names = list(artifacts.summary['label_names'])
reference_label = 'neutral'
reference_idx = label_names.index(reference_label)
label_ids = artifacts.true_label_ids
split_array = metadata['split'].to_numpy()
train_mask = split_array == 'train'
val_mask = split_array == 'val'
test_mask = split_array == 'test'

final_layer_idx = int(artifacts.layer_embeddings.shape[1] - 1)
final_layer_embeddings = artifacts.layer_embeddings[:, final_layer_idx]

# Build directions with nuisance controls (actor + statement centering)
centered_embeddings, centered_centroids, centered_directions = build_train_directions_with_controls(
    embeddings=final_layer_embeddings,
    metadata=metadata,
    label_ids=label_ids,
    label_names=label_names,
    reference_label=reference_label,
)

# Also build raw (uncontrolled) directions for comparison
raw_centroids = compute_class_centroids(final_layer_embeddings[train_mask], label_ids[train_mask], len(label_names))
raw_directions = build_direction_vectors(raw_centroids, reference_idx)

print(f'Loaded {len(metadata)} samples, {len(label_names)} classes')
print(f'Embedding dim: {final_layer_embeddings.shape[1]}')
print(f'Best layer: {final_layer_idx}')

## Direction-Only vs Trained Classifier

We evaluate two variants of the direction-only classifier:
1. **Raw directions** — built from uncentered train centroids
2. **Controlled directions** — built after actor + statement centering (nuisance control)

Both are compared against the trained linear head's test performance.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# --- Raw direction classifier ---
raw_val_result = evaluate_direction_classifier(
    final_layer_embeddings[val_mask], label_ids[val_mask], raw_directions, label_names, split_name='val',
)
raw_test_result = evaluate_direction_classifier(
    final_layer_embeddings[test_mask], label_ids[test_mask], raw_directions, label_names, split_name='test',
)

# --- Controlled direction classifier ---
ctrl_val_result = evaluate_direction_classifier(
    centered_embeddings[val_mask], label_ids[val_mask], centered_directions, label_names, split_name='val',
)
ctrl_test_result = evaluate_direction_classifier(
    centered_embeddings[test_mask], label_ids[test_mask], centered_directions, label_names, split_name='test',
)

# --- Trained head baseline ---
trained_preds = artifacts.pred_label_ids[test_mask]
trained_labels = label_ids[test_mask]
trained_metrics = {
    'accuracy': float(accuracy_score(trained_labels, trained_preds)),
    'macro_f1': float(f1_score(trained_labels, trained_preds, average='macro')),
    'weighted_f1': float(f1_score(trained_labels, trained_preds, average='weighted')),
}

comparison_rows = [
    {'method': 'direction_only (raw)', 'split': 'val', **{k: raw_val_result[k] for k in ['accuracy', 'macro_f1', 'weighted_f1']}},
    {'method': 'direction_only (raw)', 'split': 'test', **{k: raw_test_result[k] for k in ['accuracy', 'macro_f1', 'weighted_f1']}},
    {'method': 'direction_only (controlled)', 'split': 'val', **{k: ctrl_val_result[k] for k in ['accuracy', 'macro_f1', 'weighted_f1']}},
    {'method': 'direction_only (controlled)', 'split': 'test', **{k: ctrl_test_result[k] for k in ['accuracy', 'macro_f1', 'weighted_f1']}},
    {'method': 'trained_head', 'split': 'test', **trained_metrics},
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(local_output_dir / 'direction_vs_trained_comparison.csv', index=False)

display(comparison_df.round(4))

recovery_pct = (ctrl_test_result['macro_f1'] / trained_metrics['macro_f1']) * 100
print(f'\nDirection-only (controlled) recovers {recovery_pct:.1f}% of trained head Macro F1')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Per-class F1 comparison
report_dir = ctrl_test_result['classification_report']
per_class_df = pd.DataFrame([
    {
        'emotion': name,
        'direction_only_f1': report_dir[name]['f1-score'],
        'trained_head_f1': float(f1_score(
            (trained_labels == idx).astype(int),
            (trained_preds == idx).astype(int),
            zero_division=0,
        )),
    }
    for idx, name in enumerate(label_names)
])
per_class_df.to_csv(local_output_dir / 'per_class_f1_comparison.csv', index=False)
display(per_class_df.round(4))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(label_names))
width = 0.35
ax.bar([i - width/2 for i in x], per_class_df['direction_only_f1'], width, label='Direction-Only', color='steelblue')
ax.bar([i + width/2 for i in x], per_class_df['trained_head_f1'], width, label='Trained Head', color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(label_names)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1: Direction-Only vs Trained Classifier')
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig(local_output_dir / 'per_class_f1_comparison.png', dpi=220)
plt.show()

In [ ]:
# Confusion matrix for direction-only classifier
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, preds_arr) in zip(axes, [
    ('Direction-Only (Controlled)', ctrl_test_result['predictions']),
    ('Trained Head', trained_preds),
]):
    cm = confusion_matrix(label_ids[test_mask], preds_arr, labels=list(range(len(label_names))))
    disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title)

plt.suptitle('Confusion Matrices: Direction-Only vs Trained Head (Test Set)', y=1.02)
plt.tight_layout()
plt.savefig(local_output_dir / 'confusion_matrix_comparison.png', dpi=220)
plt.show()

## Layer-Wise Direction Classification

Does the direction-only classifier improve at deeper layers — mirroring the layer probing results from the centroid classifier?

In [ ]:
from src.analysis.emotion_vectors import compute_class_centroids, build_direction_vectors

layerwise_rows = []
num_layers = artifacts.layer_embeddings.shape[1]
for layer_idx in range(num_layers):
    emb = artifacts.layer_embeddings[:, layer_idx]
    centroids = compute_class_centroids(emb[train_mask], label_ids[train_mask], len(label_names))
    directions = build_direction_vectors(centroids, reference_idx)
    for split_name, mask in [('val', val_mask), ('test', test_mask)]:
        result = evaluate_direction_classifier(emb[mask], label_ids[mask], directions, label_names, split_name=split_name)
        layerwise_rows.append({'layer': layer_idx, 'split': split_name, 'macro_f1': result['macro_f1'], 'accuracy': result['accuracy']})

layerwise_df = pd.DataFrame(layerwise_rows)
layerwise_df.to_csv(local_output_dir / 'layerwise_direction_classification.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
for split_name in ['val', 'test']:
    subset = layerwise_df[layerwise_df['split'] == split_name]
    ax.plot(subset['layer'], subset['macro_f1'], marker='o', label=f'{split_name} Macro F1')
ax.set_xlabel('Transformer Layer')
ax.set_ylabel('Macro F1')
ax.set_title('Direction-Only Classification Across Layers')
ax.legend()
ax.set_xticks(range(num_layers))
plt.tight_layout()
plt.savefig(local_output_dir / 'layerwise_direction_f1.png', dpi=220)
plt.show()

In [ ]:
# --- Drive backup ---
if not IS_COLAB:
    print('Google Drive sync is intended for Colab runtimes. Skipping.')
elif drive_output_dir is None:
    print('Drive output directory is not configured.')
else:
    import shutil
    if drive_output_dir.exists():
        shutil.rmtree(drive_output_dir)
    drive_output_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output_dir, drive_output_dir)
    print('Backed up to:', drive_output_dir)

print('\nOutput files:')
for p in sorted(local_output_dir.iterdir()):
    print(f'  {p.name}')